## 1. Initialize Project Environment
Import libraries for module detection, clustering, and hub gene identification.

In [12]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import networkx as nx
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
print("numpy", np.__version__)
print("networkx", nx.__version__)

pandas 2.2.3
numpy 2.1.3
networkx 3.3


## 2. Define Configuration Parameters
Centralize module detection parameters including clustering method and minimum module size.

In [13]:
@dataclass
class ModuleConfig:
    adjacency_file: Path = Path("artifacts/task2_adjacency_matrix.csv")
    correlation_file: Path = Path("artifacts/task2_correlation_matrix.csv")
    true_modules_file: Path = Path("artifacts/task1_true_modules.csv")
    export_dir: Path = Path("artifacts")
    linkage_method: str = "average"  # WGCNA uses average linkage
    min_module_size: int = 10
    cut_height: float = 0.9  # Distance threshold for cutting dendrogram
    hub_percentile: float = 90  # Top percentile for hub genes

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["adjacency_file"] = str(info["adjacency_file"])
        info["correlation_file"] = str(info["correlation_file"])
        info["true_modules_file"] = str(info["true_modules_file"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = ModuleConfig()
CONFIG.describe()

{'adjacency_file': 'artifacts/task2_adjacency_matrix.csv',
 'correlation_file': 'artifacts/task2_correlation_matrix.csv',
 'true_modules_file': 'artifacts/task1_true_modules.csv',
 'export_dir': 'artifacts',
 'linkage_method': 'average',
 'min_module_size': 10,
 'cut_height': 0.9,
 'hub_percentile': 90}

## 3. Load Adjacency and Correlation Matrices
Load the network data from Task 2.

In [14]:
def load_matrix(path: Path) -> pd.DataFrame:
    """Load matrix with gene names as index/columns."""
    df = pd.read_csv(path, index_col=0)
    logging.info(f"Loaded matrix: {df.shape[0]} x {df.shape[1]} from {path.name}")
    return df


adjacency = load_matrix(CONFIG.adjacency_file)
correlation = load_matrix(CONFIG.correlation_file)

# Load true modules for validation
true_modules = pd.read_csv(CONFIG.true_modules_file)
logging.info(f"Loaded {len(true_modules)} true module assignments")

[INFO] Loaded matrix: 67 x 67 from task2_adjacency_matrix.csv
[INFO] Loaded matrix: 67 x 67 from task2_correlation_matrix.csv
[INFO] Loaded 200 true module assignments


## 4. Compute Topological Overlap Matrix (TOM)
TOM measures interconnectedness between genes, making module detection more robust.

In [15]:
def compute_tom(adj: pd.DataFrame) -> pd.DataFrame:
    """
    Compute Topological Overlap Matrix (TOM).
    TOM_ij = (sum_u(a_iu * a_uj) + a_ij) / (min(k_i, k_j) + 1 - a_ij)
    where k_i is the connectivity of gene i.
    """
    A = adj.values
    n = A.shape[0]

    # Connectivity: sum of adjacencies for each gene
    k = A.sum(axis=1)

    # Compute numerator: l_ij = sum_u(a_iu * a_uj)
    L = A @ A

    # TOM calculation
    TOM = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            if i == j:
                TOM[i, j] = 1.0
            else:
                numerator = L[i, j] + A[i, j]
                denominator = min(k[i], k[j]) + 1 - A[i, j]
                if denominator > 0:
                    TOM[i, j] = numerator / denominator
                    TOM[j, i] = TOM[i, j]

    tom_df = pd.DataFrame(TOM, index=adj.index, columns=adj.columns)
    logging.info(f"Computed TOM: {tom_df.shape}")
    logging.info(f"  Mean TOM: {tom_df.values[np.triu_indices(n, k=1)].mean():.4f}")

    return tom_df


tom = compute_tom(adjacency)
tom.iloc[:6, :6]

[INFO] Computed TOM: (67, 67)
[INFO]   Mean TOM: 0.2916


,TP53,MDM2,BAX,PUMA,NOXA,BID
TP53,1.000000,0.621447,0.602427,0.592194,0.611961,0.600360
MDM2,0.621447,1.000000,0.631686,0.620758,0.620945,0.631549
BAX,0.602427,0.631686,1.000000,0.552912,0.618770,0.540855
PUMA,0.592194,0.620758,0.552912,1.000000,0.610944,0.551938
NOXA,0.611961,0.620945,0.618770,0.610944,1.000000,0.624643
BID,0.600360,0.631549,0.540855,0.551938,0.624643,1.000000


## 5. Hierarchical Clustering for Module Detection
Use average linkage clustering on TOM dissimilarity (1 - TOM).

In [16]:
def detect_modules_hierarchical(
    tom: pd.DataFrame,
    method: str = "average",
    cut_height: float = 0.9,
    min_module_size: int = 10,
) -> Tuple[pd.DataFrame, np.ndarray]:
    """
    Detect modules using hierarchical clustering on TOM dissimilarity.
    """
    # Convert TOM to dissimilarity
    dissim = 1 - tom.values
    np.fill_diagonal(dissim, 0)

    # Convert to condensed distance matrix
    dissim_condensed = squareform(dissim)

    # Hierarchical clustering
    Z = linkage(dissim_condensed, method=method)

    # Cut dendrogram at specified height
    clusters = fcluster(Z, t=cut_height, criterion="distance")

    # Create module assignments
    module_df = pd.DataFrame({"Gene": tom.index, "Module": clusters})

    # Filter small modules (assign to module 0 = unassigned)
    module_sizes = module_df["Module"].value_counts()
    small_modules = module_sizes[module_sizes < min_module_size].index
    module_df.loc[module_df["Module"].isin(small_modules), "Module"] = 0

    # Renumber modules
    unique_modules = sorted(module_df["Module"].unique())
    module_map = {old: new for new, old in enumerate(unique_modules)}
    module_df["Module"] = module_df["Module"].map(module_map)

    n_modules = len(module_df["Module"].unique())
    logging.info(f"Detected {n_modules} modules (including unassigned)")
    logging.info(
        f"Module sizes: {module_df['Module'].value_counts().sort_index().to_dict()}"
    )

    return module_df, Z


modules_hier, linkage_matrix = detect_modules_hierarchical(
    tom,
    method=CONFIG.linkage_method,
    cut_height=CONFIG.cut_height,
    min_module_size=CONFIG.min_module_size,
)
modules_hier.head(10)

[INFO] Detected 2 modules (including unassigned)
[INFO] Module sizes: {0: 7, 1: 60}


,Gene,Module
0,TP53,1
1,MDM2,1
2,BAX,1
3,PUMA,1
4,NOXA,1
5,BID,1
6,APAF1,1
7,CASP9,1
8,CASP3,1
9,BCL2,1


In [17]:
# Module size distribution
print("Module size distribution:")
modules_hier["Module"].value_counts().sort_index()

Module size distribution:


Module
0     7
1    60
Name: count, dtype: int64

## 6. Alternative: Louvain Community Detection
Use NetworkX's community detection for comparison.

In [18]:
def detect_modules_louvain(
    adj: pd.DataFrame, edge_threshold: float = 0.01
) -> pd.DataFrame:
    """
    Detect modules using Louvain community detection.
    """
    # Create graph from adjacency matrix
    G = nx.Graph()
    G.add_nodes_from(adj.index)

    # Add edges above threshold
    for i, gene_i in enumerate(adj.index):
        for j, gene_j in enumerate(adj.columns):
            if i < j and adj.iloc[i, j] > edge_threshold:
                G.add_edge(gene_i, gene_j, weight=adj.iloc[i, j])

    logging.info(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # Remove isolated nodes
    isolates = list(nx.isolates(G))
    if isolates:
        logging.info(f"Removing {len(isolates)} isolated nodes")
        G.remove_nodes_from(isolates)

    # Louvain community detection
    try:
        communities = nx.community.louvain_communities(G, seed=42, weight="weight")
        method = "Louvain"
    except AttributeError:
        communities = nx.community.greedy_modularity_communities(G, weight="weight")
        method = "Greedy Modularity"

    logging.info(f"Detected {len(communities)} communities using {method}")

    # Create module assignments
    gene_to_module = {}
    for module_id, community in enumerate(communities):
        for gene in community:
            gene_to_module[gene] = module_id + 1  # Module 0 reserved for unassigned

    # Add unassigned genes
    for gene in adj.index:
        if gene not in gene_to_module:
            gene_to_module[gene] = 0

    module_df = pd.DataFrame(
        [{"Gene": gene, "Module": mod} for gene, mod in gene_to_module.items()]
    )

    return module_df


modules_louvain = detect_modules_louvain(adjacency)
print("\nLouvain module sizes:")
modules_louvain["Module"].value_counts().sort_index()

[INFO] Graph: 67 nodes, 1791 edges
[INFO] Detected 4 communities using Louvain



Louvain module sizes:


Module
1    20
2    20
3    20
4     7
Name: count, dtype: int64

## 7. Identify Hub Genes
Hub genes have high intramodular connectivity - they are central to their module.

In [19]:
def identify_hub_genes(
    adj: pd.DataFrame, modules: pd.DataFrame, percentile: float = 90
) -> pd.DataFrame:
    """
    Identify hub genes based on intramodular connectivity.
    Hub genes are in the top percentile of connectivity within their module.
    """
    # Calculate global connectivity
    connectivity = adj.sum(axis=1)

    # Merge connectivity with module assignments
    hub_df = modules.copy()
    hub_df["Connectivity"] = hub_df["Gene"].map(connectivity)

    # Calculate intramodular connectivity
    intramodular_conn = []
    for _, row in hub_df.iterrows():
        gene = row["Gene"]
        module = row["Module"]

        # Get genes in same module
        module_genes = hub_df[hub_df["Module"] == module]["Gene"].tolist()

        # Sum connectivity to module members
        if gene in adj.index:
            intra_conn = adj.loc[
                gene, [g for g in module_genes if g in adj.columns]
            ].sum()
        else:
            intra_conn = 0

        intramodular_conn.append(intra_conn)

    hub_df["IntramodularConnectivity"] = intramodular_conn

    # Identify hub genes per module
    hub_df["IsHub"] = False
    for module in hub_df["Module"].unique():
        if module == 0:  # Skip unassigned
            continue

        module_mask = hub_df["Module"] == module
        threshold = np.percentile(
            hub_df.loc[module_mask, "IntramodularConnectivity"], percentile
        )
        hub_df.loc[
            module_mask & (hub_df["IntramodularConnectivity"] >= threshold), "IsHub"
        ] = True

    n_hubs = hub_df["IsHub"].sum()
    logging.info(
        f"Identified {n_hubs} hub genes (top {100 - percentile:.0f}% connectivity)"
    )

    return hub_df


hub_analysis = identify_hub_genes(
    adjacency,
    modules_louvain,  # Use Louvain modules for hub analysis
    percentile=CONFIG.hub_percentile,
)

# Show hub genes
print("\nHub genes by module:")
hub_analysis[hub_analysis["IsHub"]].sort_values(
    ["Module", "IntramodularConnectivity"], ascending=[True, False]
)

[INFO] Identified 7 hub genes (top 10% connectivity)



Hub genes by module:


,Gene,Module,Connectivity,IntramodularConnectivity,IsHub
11,NOXA,1,24.603212,14.529462,True
1,APAF1,1,24.394018,14.398655,True
34,E2F2,2,24.021133,15.069789,True
22,CDKN1A,2,24.944646,15.063546,True
46,XRCC1,3,12.713441,10.346503,True
52,ERCC2,3,11.371611,9.907703,True
66,AMPK,4,2.347907,2.344947,True


In [20]:
# Compare detected modules with true modules
def compare_modules(detected: pd.DataFrame, true_df: pd.DataFrame) -> pd.DataFrame:
    """Compare detected modules with ground truth."""
    merged = detected.merge(true_df, on="Gene", how="left")

    # Cross-tabulation
    crosstab = pd.crosstab(merged["Module"], merged["TrueModule"], margins=True)
    return crosstab


comparison = compare_modules(modules_louvain, true_modules)
print("\nModule comparison (Detected vs True):")
comparison


Module comparison (Detected vs True):


TrueModule,Cell_cycle,DNA_damage,Metabolic,TP53_pathway,All
Module,,,,,
1,0,0,0,20,20
2,20,0,0,0,20
3,0,20,0,0,20
4,0,0,7,0,7
All,20,20,7,20,67


## 8. Validate with Unit Tests
Sanity checks for module detection and hub identification.

In [21]:
def test_all_genes_assigned():
    """All genes should have a module assignment."""
    assert len(modules_louvain) == len(adjacency), "Missing gene assignments"


def test_hub_in_module():
    """Hub genes should belong to non-zero modules."""
    hubs = hub_analysis[hub_analysis["IsHub"]]
    assert (hubs["Module"] != 0).all(), "Hub in unassigned module"


def test_tom_symmetric():
    """TOM should be symmetric."""
    assert np.allclose(tom.values, tom.values.T), "TOM not symmetric"


test_all_genes_assigned()
test_hub_in_module()
test_tom_symmetric()
print("All module detection tests passed.")

All module detection tests passed.


## 9. Export Results
Save module assignments, hub genes, and TOM for downstream analysis.

In [22]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save module assignments (using Louvain as primary)
modules_louvain.to_csv(EXPORT_DIR / "task3_module_assignments.csv", index=False)
print(
    f"[OK] Module assignments saved to: {EXPORT_DIR / 'task3_module_assignments.csv'}"
)

# Save hierarchical modules for comparison
modules_hier.to_csv(EXPORT_DIR / "task3_module_hierarchical.csv", index=False)
print(
    f"[OK] Hierarchical modules saved to: {EXPORT_DIR / 'task3_module_hierarchical.csv'}"
)

# Save hub gene analysis
hub_analysis.to_csv(EXPORT_DIR / "task3_hub_genes.csv", index=False)
print(f"[OK] Hub gene analysis saved to: {EXPORT_DIR / 'task3_hub_genes.csv'}")

# Save TOM
tom.to_csv(EXPORT_DIR / "task3_tom_matrix.csv")
print(f"[OK] TOM matrix saved to: {EXPORT_DIR / 'task3_tom_matrix.csv'}")

# Save module comparison
comparison.to_csv(EXPORT_DIR / "task3_module_comparison.csv")
print(f"[OK] Module comparison saved to: {EXPORT_DIR / 'task3_module_comparison.csv'}")

# Summary stats
summary = {
    "n_modules_louvain": len(modules_louvain["Module"].unique()),
    "n_modules_hierarchical": len(modules_hier["Module"].unique()),
    "n_hub_genes": hub_analysis["IsHub"].sum(),
    "largest_module_size": modules_louvain["Module"].value_counts().max(),
}
pd.DataFrame([summary]).to_csv(EXPORT_DIR / "task3_summary.csv", index=False)
print(f"[OK] Summary saved to: {EXPORT_DIR / 'task3_summary.csv'}")

[OK] Module assignments saved to: artifacts/task3_module_assignments.csv
[OK] Hierarchical modules saved to: artifacts/task3_module_hierarchical.csv
[OK] Hub gene analysis saved to: artifacts/task3_hub_genes.csv
[OK] TOM matrix saved to: artifacts/task3_tom_matrix.csv
[OK] Module comparison saved to: artifacts/task3_module_comparison.csv
[OK] Summary saved to: artifacts/task3_summary.csv
